In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
from pathlib import Path

img = Path(r"C:\Users\Admin\Desktop\Grad_Project\Code\Datasets\bdd100k_images_100k\100k\train")
lbl = Path(r"C:\Users\Admin\Desktop\Grad_Project\Code\Datasets\BDD100k_V2\labels\train")

imgs = list(img.glob("*.jpg"))
lbls = list(lbl.glob("*.txt"))

print(f"Images: {len(imgs):,}")
print(f"Labels: {len(lbls):,}")
print(f"Sample image: {imgs[0].stem}")
print(f"Matching label exists: {(lbl / (imgs[0].stem + '.txt')).exists()}")

Images: 70,000
Labels: 70,000
Sample image: 0000f77c-6257be58
Matching label exists: True


In [3]:
"""
Tamakkan - YOLOv11s Training Script (v2 - high-res + tuned augs)
=================================================================
Dataset  : BDD100K full 100k, 7 classes
Model    : YOLOv11s (small, real-time capable on-device)
Target   : push mAP50 from 0.63 -> 0.70+ via higher resolution
           and driving-appropriate augmentations
 
Changes from v1:
  - imgsz: 640 -> 1280  (biggest lever for small objects: VRU, person, lights)
  - batch: 16 -> 8      (fit 1280 in 16GB VRAM)
  - flipud: 0.01 -> 0.0 (cars don't appear upside-down on roads)
  - degrees: 10 -> 5    (dashcams don't tilt that much)
  - mixup: 0.1 -> 0.0   (hurts small-object localization)
  - copy_paste: 0.3 -> 0.5 (more synthetic VRU/person instances)
  - patience: 20 -> 30  (1280 converges slower, give it room)
  - epochs: 100 -> 120  (room for slower convergence)
"""
 
from ultralytics import YOLO
from pathlib import Path
import torch

In [4]:
# ── VERIFY GPU ─────────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "CUDA not available — check PyTorch installation"
print(f"Training on: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\n")

Training on: NVIDIA GeForce RTX 5080
VRAM: 17.1 GB



In [5]:
# ── PATHS ──────────────────────────────────────────────────────────────────────
DATA_YAML  = r"C:\Users\Admin\Desktop\Grad_Project\Code\Datasets\BDD100k_V2\data.yaml"
OUTPUT_DIR = r"C:\Users\Admin\Desktop\Grad_Project\Code\Yolov11s_training_Results"

In [6]:
# ── LOAD MODEL ─────────────────────────────────────────────────────────────────
# Loads YOLOv11s pretrained on COCO — we fine-tune from here, not scratch
model = YOLO("yolo11s.pt")

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────────────
results = model.train(
    # --- Core ---
    data        = DATA_YAML,
    epochs      = 120,         # ↑ from 100, more room at higher res
    imgsz       = 1280,        # ↑ from 640 — main driver of improvement
    batch       = 8,           # ↓ from 16, to fit 1280 in 16GB VRAM
    device      = 0,
 
    # --- Output ---
    project     = OUTPUT_DIR,
    name        = "tamakkan_v2_hires",
    exist_ok    = False,
 
    # --- Optimization (unchanged) ---
    optimizer   = "AdamW",
    lr0         = 0.001,
    lrf         = 0.01,
    momentum    = 0.937,
    weight_decay= 0.0005,
    cos_lr      = True,
    warmup_epochs    = 3,
    warmup_momentum  = 0.8,
 
    # --- Loss weighting (unchanged) ---
    cls         = 0.7,
 
    # --- Augmentation (tuned for dashcam data) ---
    mosaic      = 1.0,
    copy_paste  = 0.5,         # ↑ from 0.3 — more rare-class instances
    mixup       = 0.0,         # ↓ from 0.1 — off, hurts small object boxes
    degrees     = 5.0,         # ↓ from 10 — realistic for dashcam
    translate   = 0.1,
    scale       = 0.5,
    shear       = 2.0,
    perspective = 0.0,
    flipud      = 0.0,         # ↓ from 0.01 — off, cars aren't upside down
    fliplr      = 0.5,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
 
    # --- Early stopping ---
    patience    = 30,          # ↑ from 20 — higher res converges slower
 
    # --- Efficiency ---
    workers     = 4,
    cache       = False,
    amp         = True,
    close_mosaic= 15,          # ↑ from 10 — more stable final epochs
 
    # --- Logging ---
    plots       = True,
    save        = True,
    save_period = 10,
    verbose     = True,
)
 
print("\nTraining complete.")
print(f"Results saved to: {OUTPUT_DIR}")
print(f"Best weights: {OUTPUT_DIR}/tamakkan_v2_hires/weights/best.pt")

Ultralytics 8.4.37  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.7, cls_pw=0.0, compile=False, conf=None, copy_paste=0.5, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Admin\Desktop\Grad_Project\Code\Datasets\BDD100k_V2\data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tamakkan_v2_hires, nbs=64, nms=False, opset=None,